In [13]:
# Core
import numpy as np
import pandas as pd
import os
import cv2
from PIL import Image

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout, BatchNormalization, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy, CategoricalCrossentropy
from tensorflow.keras.metrics import AUC, Precision, Recall
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# Transfer learning backbones
from tensorflow.keras.applications import EfficientNetB0, ResNet50, DenseNet121

# Data preprocessing & augmentation
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Evaluation
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [11]:
train_dir = r'C:\Scan_CNN\Training'
test_dir = r'C:\Scan_CNN\Testing'

def build_dataframe(data_dir):
    filepaths = []
    labels = []
    for label in os.listdir(data_dir):
        label_folder = os.path.join(data_dir, label)
        if os.path.isdir(label_folder):
            for image in os.listdir(label_folder):
                if image.endswith(('.jpg', '.jpeg', '.png')):
                    filepaths.append(os.path.join(label_folder, image))
                    labels.append(label)
    return pd.DataFrame({'filepath': filepaths, 'label': labels})

train_df = build_dataframe(train_dir)
test_df = build_dataframe(test_dir)

print("Training set:")
print(train_df['label'].value_counts())
print(f"\nTotal training images: {len(train_df)}")

print("\nTesting set:")
print(test_df['label'].value_counts())
print(f"\nTotal testing images: {len(test_df)}")

Training set:
label
glioma        1400
meningioma    1400
notumor       1400
pituitary     1400
Name: count, dtype: int64

Total training images: 5600

Testing set:
label
glioma        400
meningioma    400
notumor       400
pituitary     400
Name: count, dtype: int64

Total testing images: 1600


In [10]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Augmentation for training, just normalization for testing
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1
)

test_datagen = ImageDataGenerator(rescale=1./255)

# Load images from dataframe
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='filepath',
    y_col='label',
    target_size=(224, 224),
    batch_size=4,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='filepath',
    y_col='label',
    target_size=(224, 224),
    batch_size=4,
    class_mode='categorical',
    shuffle=False
)

print(train_generator.class_indices)

Found 5600 validated image filenames belonging to 4 classes.
Found 1600 validated image filenames belonging to 4 classes.
{'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}


In [14]:
def generate_model():
    model = tf.keras.Sequential([
        #model will start with 32 filetrs with them being 3x3 with a relu activation function
        #they will be pooled into 2x2 matrixes 
        tf.keras.layers.Conv2D(32, kernel_size=3, activation='relu'),
        tf.keras.layers.MaxPool2D(pool_size = (2,2), strides=(1,1)),

        tf.keras.layers.Conv2D(64,kernel_size=3, activation='relu'),
        tf.keras.layers.MaxPool2D(pool_size = (2,2), strides = (2,2)),

        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(1024, activation='relu'),
        tf.keras.layers.Dense(4, activation='softmax')
    ])
    return model

In [15]:
model = generate_model()

model.compile(optimizer='adam', 
              loss='categorical_crossentropy', 
              metrics=['accuracy'])

model.fit(
    train_generator,
    validation_data = test_generator,
    epochs=5
)

Epoch 1/5


ResourceExhaustedError: Graph execution error:

Detected at node StatefulPartitionedCall/gradient_tape/sequential_1_1/dense_2_1/MatMul/MatMul_1 defined at (most recent call last):
<stack traces unavailable>
OOM when allocating tensor with shape[760384,1024] and type float on /job:localhost/replica:0/task:0/device:CPU:0 by allocator mklcpu
	 [[{{node StatefulPartitionedCall/gradient_tape/sequential_1_1/dense_2_1/MatMul/MatMul_1}}]]
Hint: If you want to see a list of allocated tensors when OOM happens, add report_tensor_allocations_upon_oom to RunOptions for current allocation info. This isn't available when running in Eager mode.
 [Op:__inference_multi_step_on_iterator_3870]